In [5]:
from playwright.async_api import async_playwright

async def login_and_save_state(path="ig_state.json"):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        await page.goto("https://www.instagram.com/", wait_until="domcontentloaded")
        print("Faça login manualmente na página,  depois volte aqui")
        # give yourself time to login
        await page.wait_for_timeout(120000)
        await page.context.storage_state(path=path)
        await browser.close()

# run:
await login_and_save_state("ig_state.json")


Faça login manualmente na página,  depois volte aqui


In [8]:
from playwright.async_api import async_playwright, TimeoutError as PWTimeout

async def collect_post_links_logged_in(
    hashtag: str,
    state_path="ig_state.json",
    max_links: int = 400,
    scrolls: int = 50,
):
    url = f"https://www.instagram.com/explore/tags/{hashtag}/"
    links = set()

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            storage_state=state_path,
            viewport={"width": 1600, "height": 900},
            user_agent=(
                "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
            ),
            locale="pt-BR",
        )
        page = await context.new_page()

        # Increase navigation timeout (default is 30s)
        page.set_default_navigation_timeout(240000)
        page.set_default_timeout(240000)

        # Esperar até o documento carregar por inteiro
        await page.goto(url, wait_until="domcontentloaded")

        # Wait for either post tiles OR a login/challenge page clue
        try:
            await page.wait_for_selector("a[href*='/p/'], a[href*='/reel/']", timeout=10000)
        except PWTimeout:
            # Tirar print caso haja bloqueio
            await page.screenshot(path="debug_tag_page.png", full_page=True)
            html = await page.content()
            await browser.close()
            raise RuntimeError(
                "Could not find post links on the hashtag page. "
                "Saved debug_tag_page.png. You may be on a login wall/challenge or markup changed."
            )

        for _ in range(scrolls):
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await page.wait_for_timeout(4000)

            anchors = await page.query_selector_all("a[href*='/p/'], a[href*='/reel/']")
            for a in anchors:
                href = await a.get_attribute("href") or ""
                if href.startswith("/p/") or href.startswith("/reel/"):
                    links.add("https://www.instagram.com" + href.split("?")[0])

            if len(links) >= max_links:
                break

        await browser.close()

    return list(links)[:max_links]

post_links = await collect_post_links_logged_in("carnaval", "ig_state.json", max_links=400, scrolls=50)
len(post_links), post_links[:400]

(69,
 ['https://www.instagram.com/p/DTVaalEDsvN/',
  'https://www.instagram.com/p/DTl0TJDAKHQ/',
  'https://www.instagram.com/p/DTRIT5uFt9N/',
  'https://www.instagram.com/p/DTiky-riafU/',
  'https://www.instagram.com/p/DTf149Qjr4x/',
  'https://www.instagram.com/p/DTdVtg9jUJi/',
  'https://www.instagram.com/p/DSLOs-8knM4/',
  'https://www.instagram.com/p/DSaU0UXEaGJ/',
  'https://www.instagram.com/p/DRp7YFUkg3k/',
  'https://www.instagram.com/p/DTnuedyDsku/',
  'https://www.instagram.com/p/DS49220kVHA/',
  'https://www.instagram.com/p/DTsZA_CgG93/',
  'https://www.instagram.com/p/DTNvamlkV4K/',
  'https://www.instagram.com/p/DTNXPwrAPVK/',
  'https://www.instagram.com/p/DNRexlktveE/',
  'https://www.instagram.com/p/DTnRI4kDavQ/',
  'https://www.instagram.com/p/DTdTlobEWRe/',
  'https://www.instagram.com/p/DTtYeVuj5wq/',
  'https://www.instagram.com/p/DTkXZiKDTtK/',
  'https://www.instagram.com/p/DTBW_njEbRi/',
  'https://www.instagram.com/p/DTVO__LAARc/',
  'https://www.instagram.com/

In [9]:
async def collect_links_for_hashtags(
    hashtags,
    state_path="ig_state.json",
    max_links_per_hashtag=400,
    scrolls=50,
    dedupe_global=True,
):
    results = {}          # hashtag -> list of links
    all_links = set()     # global dedupe (optional)

    for tag in hashtags:
        tag = tag.lstrip("#").strip()
        print(f"\n=== Collecting #{tag} ===")

        links = await collect_post_links_logged_in(
            tag,
            state_path=state_path,
            max_links=max_links_per_hashtag,
            scrolls=scrolls,
        )

        if dedupe_global:
            # remove links already seen from previous hashtags
            new_links = [u for u in links if u not in all_links]
            all_links.update(new_links)
            results[tag] = new_links
            print(f"#{tag}: {len(new_links)} new links (of {len(links)} collected)")
        else:
            results[tag] = links
            print(f"#{tag}: {len(links)} links collected")

    # If you want a single list for downstream code:
    post_links = list(all_links) if dedupe_global else [u for tag in results for u in results[tag]]
    return results, post_links


In [10]:
hashtags = ["carnaval", "carnavalrecife", "recife"]  # add as many as you want

links_by_hashtag, post_links = await collect_links_for_hashtags(
    hashtags,
    state_path="ig_state.json",
    max_links_per_hashtag=400,
    scrolls=50,
    dedupe_global=True,
)

len(post_links), post_links[:50]


=== Collecting #carnaval ===
#carnaval: 78 new links (of 78 collected)

=== Collecting #carnavalrecife ===
#carnavalrecife: 79 new links (of 80 collected)

=== Collecting #recife ===
#recife: 76 new links (of 82 collected)


(233,
 ['https://www.instagram.com/p/DMVYZNuuiAT/',
  'https://www.instagram.com/p/DS49220kVHA/',
  'https://www.instagram.com/p/DTtqHjfEYUJ/',
  'https://www.instagram.com/p/DR2JsOhEaf2/',
  'https://www.instagram.com/p/DSKskT-EbOG/',
  'https://www.instagram.com/p/DGox0_6Rq5J/',
  'https://www.instagram.com/p/DTnRI4kDavQ/',
  'https://www.instagram.com/p/DTOADPgiQag/',
  'https://www.instagram.com/p/DSTAEWMAX3k/',
  'https://www.instagram.com/p/DSxb64NDdj7/',
  'https://www.instagram.com/p/DTK6obNEYBH/',
  'https://www.instagram.com/p/DTVO__LAARc/',
  'https://www.instagram.com/p/DS3VZ6kDZTh/',
  'https://www.instagram.com/p/DS4wZc5jJmw/',
  'https://www.instagram.com/p/DTfeqigkXWY/',
  'https://www.instagram.com/p/DSm411tAU50/',
  'https://www.instagram.com/p/DS7d06Okexg/',
  'https://www.instagram.com/p/DSawzGUEkQ3/',
  'https://www.instagram.com/p/DNoif7xBNY2/',
  'https://www.instagram.com/p/DTbM6zXEWMV/',
  'https://www.instagram.com/p/DTC4K4njRrM/',
  'https://www.instagram.com

In [11]:
post_links

['https://www.instagram.com/p/DMVYZNuuiAT/',
 'https://www.instagram.com/p/DS49220kVHA/',
 'https://www.instagram.com/p/DTtqHjfEYUJ/',
 'https://www.instagram.com/p/DR2JsOhEaf2/',
 'https://www.instagram.com/p/DSKskT-EbOG/',
 'https://www.instagram.com/p/DGox0_6Rq5J/',
 'https://www.instagram.com/p/DTnRI4kDavQ/',
 'https://www.instagram.com/p/DTOADPgiQag/',
 'https://www.instagram.com/p/DSTAEWMAX3k/',
 'https://www.instagram.com/p/DSxb64NDdj7/',
 'https://www.instagram.com/p/DTK6obNEYBH/',
 'https://www.instagram.com/p/DTVO__LAARc/',
 'https://www.instagram.com/p/DS3VZ6kDZTh/',
 'https://www.instagram.com/p/DS4wZc5jJmw/',
 'https://www.instagram.com/p/DTfeqigkXWY/',
 'https://www.instagram.com/p/DSm411tAU50/',
 'https://www.instagram.com/p/DS7d06Okexg/',
 'https://www.instagram.com/p/DSawzGUEkQ3/',
 'https://www.instagram.com/p/DNoif7xBNY2/',
 'https://www.instagram.com/p/DTbM6zXEWMV/',
 'https://www.instagram.com/p/DTC4K4njRrM/',
 'https://www.instagram.com/p/DR2NLgZDlTC/',
 'https://

In [12]:
from scrapegraphai.graphs import SmartScraperGraph

PROMPT = """
You are extracting data from an Instagram post page.

Return ONLY valid JSON with these keys:
- username: the author's Instagram handle (without @)
- text: the post caption text (empty string if missing)
- location: location title (NA if missing)

No extra keys. No explanations. JSON only.
"""

def extract_post_with_sga(url: str, ollama_model="ollama/functiongemma") -> dict:
    graph_config = {
        "llm": {
            "model": ollama_model,   # functiongemma via Ollama
            "temperature": 0.0,
            "format": "json",
            "model_tokens": 4096,    # set explicitly to avoid default warning (tune 4096–8192)
        },
        "verbose": False,
        "headless": True,
    }

    graph = SmartScraperGraph(
        prompt=PROMPT,
        source=url,
        config=graph_config,
    )
    return graph.run()


In [20]:
#this section covers with async turned off

In [13]:
from playwright.async_api import async_playwright
import asyncio

async def fetch_html_logged_in(url: str, state_path="ig_state.json") -> str:
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(storage_state=state_path)
        page = await context.new_page()

        await page.goto(url, wait_until="domcontentloaded")
        await page.wait_for_timeout(1500)  # let it settle a bit
        html = await page.content()

        await browser.close()
        return html

In [14]:
from scrapegraphai.graphs import SmartScraperGraph

PROMPT = """
Extract the Instagram post data from the provided HTML.

Return ONLY valid JSON with:
- username: the author's handle (without @)
- text: the post caption text (empty string if missing)
- location: location title (NA if missing)


JSON only. No commentary.
"""

def extract_from_html_with_sga(html: str, model="ollama/ministral-3:3b") -> dict:
    graph_config = {
        "llm": {
            "model": model,          # gemma3:12b via Ollama
            "temperature": 0.0,
            "format": "json",
            "model_tokens": 4096,    # set explicitly; tune 4096–8192 for speed vs robustness
        },
        "verbose": False,
        "headless": True,  # irrelevant when source is HTML, but harmless
    }

    graph = SmartScraperGraph(
        prompt=PROMPT,
        source=html,       # HTML string, not URL
        config=graph_config,
    )
    return graph.run()


In [ ]:
#Faster, might get rate-limited

In [15]:
# Jupyter-ready: Instagram post -> HTML (Playwright logged-in) -> Extract (ScrapeGraphAI + Ollama ministral-3:3b) -> CSV
# Assumes you already have: post_links = [...] and ig_state.json in the working directory.

import asyncio
import pandas as pd
from playwright.async_api import async_playwright
from scrapegraphai.graphs import SmartScraperGraph

MODEL = "ollama/ministral-3:3b"

PROMPT = """
Return ONLY valid JSON with:
- username: the author's handle
- text: the post caption text (empty string if missing)
- location: the post city region
JSON only.
"""

def extract_from_html_with_sga(html: str, model: str = MODEL) -> dict:
    graph_config = {
        "llm": {
            "model": model,
            "temperature": 0.0,
            "format": "json",
            "model_tokens": 4096,  # increase to 8192 if you see truncation
        },
        "verbose": False,
        "headless": True,
    }
    graph = SmartScraperGraph(prompt=PROMPT, source=html, config=graph_config)
    return graph.run()

async def scrape_posts_with_ministral(
    post_links,
    state_path="ig_state.json",
    out_csv="carnaval_posts_ministral_3_3b.csv",
    max_n=None,
    concurrency=2,
    nav_timeout_ms=60000,
    settle_ms=2500,
):
    urls = post_links[:max_n] if max_n else list(post_links)
    rows = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(storage_state=state_path)

        # Block heavy resources to speed up
        async def route_handler(route):
            rt = route.request.resource_type
            if rt in ("image", "media", "font"):
                await route.abort()
            else:
                await route.continue_()

        await context.route("**/*", route_handler)

        sem = asyncio.Semaphore(concurrency)

        async def process_one(url: str):
            async with sem:
                page = await context.new_page()
                try:
                    await page.goto(url, wait_until="domcontentloaded", timeout=nav_timeout_ms)
                    await page.wait_for_timeout(settle_ms)
                    html = await page.content()

                    data = extract_from_html_with_sga(html, model=MODEL)
                    content = data.get("content", data)

                    return {
                        "url": url,
                        "username": (content.get("username") or "").strip(),
                        "text": (content.get("text") or "").strip(),
                        "location": (content.get("location") or "").strip(),
                        "error": "",
                        
                    }
                except Exception as e:
                    return {"url": url, "username": "", "text": "", "location":"", "error": str(e)}
                finally:
                    await page.close()

        tasks = [process_one(u) for u in urls]

        for idx, coro in enumerate(asyncio.as_completed(tasks), start=1):
            row = await coro
            rows.append(row)
            if row["error"]:
                print(f"[{idx}/{len(urls)}] FAIL: {row['error'][:140]}")
            else:
                print(f"[{idx}/{len(urls)}] OK")

        await browser.close()

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False, encoding="utf-8")
    return df

df = await scrape_posts_with_ministral(
    post_links,
    state_path="ig_state.json",
    out_csv="carnaval_posts_ministral_3_3b.csv",
    max_n=None,
    concurrency=2,
)

df.head()



{'content': {'username': 'paulopaivafoto', 'text': 'A lua cheia mais próxima da Terra neste ano iluminou o céu do Recife e a Ilha do Retiro. Fenômeno conhecido como Superlua, ela aparece até 14% maior e 30% mais brilhante do que o normal. 🌕', 'location': 'Ilha do Retiro, Recife'}}
✨ Try enhanced version of ScrapegraphAI at ]8;;https://scrapegraphai.com\https://scrapegraphai.com]8;;\ ✨
[1/233] OK
{'content': {'username': 'giroblog', 'text': 'Emoji não disponível para tradução direta, mas o post menciona: \'E não é que subi com ela no elevador da Esmape? Foi um prazer conhecê-la. Sucesso para o seu sobrinho. Sei que ele vai adorar o meu "país" Pernambuco!\'', 'location': 'Recife'}}
✨ Try enhanced version of ScrapegraphAI at ]8;;https://scrapegraphai.com\https://scrapegraphai.com]8;;\ ✨
[2/233] OK
{'content': [{'username': 'dornelas_1221', 'text': 'Bloco da zn vai ser aonde?', 'location': 'NA'}, {'username': 'jpedro_melo05', 'text': 'Se organizar direito da pra ir em todos KKKKKKK

,url,username,text,location,error
0,https://www.instagram.com/p/DQsj9--DZ-t/,paulopaivafoto,A lua cheia mais próxima da Terra neste ano il...,"Ilha do Retiro, Recife",
1,https://www.instagram.com/p/DTOADPgiQag/,giroblog,"Emoji não disponível para tradução direta, mas...",Recife,
2,https://www.instagram.com/p/DTVaalEDsvN/,,,,'list' object has no attribute 'get'
3,https://www.instagram.com/p/DTfdpyEFX22/,tecodeantonio,"Quem tem pena, quem tem dó? Carnaval é devoção...",Zona Leste de São Paulo,
4,https://www.instagram.com/p/DTNkwvYEVkh/,familiartpresentes,"Eita que tá chegando... Se liga, aqui na [@fam...",Recife,


In [2]:
import pandas as pd
import requests
import json
import time

in_path = "carnaval_posts_ministral_3_3b.csv"  # ajuste se necessário
out_path = "carnaval_posts_with_risk.csv"

df = pd.read_csv(in_path)

# garanta colunas (ajuste nomes se no seu csv estiver diferente)
# df.columns
# esperado: "text" e "location" (se location não existir, tudo bem)
if "location" not in df.columns:
    df["location"] = ""

SYSTEM = """You are a careful public-health triage classifier.
Classify whether the text suggests an epidemiological danger/crisis/event (e.g., outbreak, many people ill, suspected contamination, mass vomiting/diarrhea, unusual cluster, hospital overload, etc.).
Do NOT diagnose individuals. Use only the text evidence. Output in Portuguese.
Return ONLY JSON: {"risk":"low|medium|high","signal":"short reason"}"""

def ollama_chat(model, system, user, timeout=120):
    # endpoint padrão do Ollama
    url = "http://localhost:11434/api/chat"
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "stream": False,
        "format": "json",
        "options": {
            "temperature": 0.0,
            # "num_ctx": 4096,  # se quiser forçar contexto
        }
    }
    r = requests.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    # Ollama retorna a resposta em data["message"]["content"]
    content = data["message"]["content"]
    return json.loads(content)

MODEL = "ministral-3:3b"  # ou "gemma3:12b", etc

risks = []
signals = []

for i, row in df.iterrows():
    text = str(row.get("text", "") or "")
    loc = str(row.get("location", "") or "")

    # diminui custo: corta HTML/hashtags exageradas
    text_short = text.strip()
    if len(text_short) > 2000:
        text_short = text_short[:2000]

    prompt = f"LOCATION: {loc}\nTEXT: {text_short}\n\nReturn JSON."

    try:
        out = ollama_chat(MODEL, SYSTEM, prompt)
        risks.append(out.get("risk", "low"))
        signals.append(out.get("signal", ""))
    except Exception as e:
        risks.append("")
        signals.append(f"error: {e}")

    # throttle leve pra não travar seu PC
    time.sleep(0.2)

df["risk"] = risks
df["risk_signal"] = signals

df.to_csv(out_path, index=False, encoding="utf-8")
df.head()


,url,username,text,location,error,risk,risk_signal
0,https://www.instagram.com/p/DQsj9--DZ-t/,paulopaivafoto,A lua cheia mais próxima da Terra neste ano il...,"Ilha do Retiro, Recife",NaN,low,Fenômeno astronômico comum (Superlua) sem rela...
1,https://www.instagram.com/p/DTOADPgiQag/,giroblog,"Emoji não disponível para tradução direta, mas...",Recife,NaN,low,"Não há menção a sintomas, contágio, doença, em..."
2,https://www.instagram.com/p/DTVaalEDsvN/,NaN,NaN,NaN,'list' object has no attribute 'get',unknown,texto ausente ou não fornecido para análise
3,https://www.instagram.com/p/DTfdpyEFX22/,tecodeantonio,"Quem tem pena, quem tem dó? Carnaval é devoção...",Zona Leste de São Paulo,NaN,low,"Não há menção a sintomas de doença, contaminaç..."
4,https://www.instagram.com/p/DTNkwvYEVkh/,familiartpresentes,"Eita que tá chegando... Se liga, aqui na [@fam...",Recife,NaN,low,Não há indícios de perigo epidemiológico ou cr...


In [11]:
df
df.to_csv('banco-semantico-jan22-2026.csv')